## Sel 1: Import Pustaka

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import mlflow
import dagshub

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [2]:
# Pastikan path ini sesuai dengan posisi .env milikmu
load_dotenv('../.env', override=True)

# Ambil URL MLflow utuh 
db_url_mlflow = os.getenv("DATABASE_URL_DIRECT")

if not db_url_mlflow:
    raise ValueError("DATABASE_URL_DIRECT tidak ditemukan di .env!")

# Buat URL polos khusus untuk Pandas & SQLAlchemy
db_url_pandas = db_url_mlflow.split("?")[0]

print("Membuka jembatan ke Supabase...")
# Gunakan URL polos untuk engine
engine = create_engine(db_url_pandas)

# ==========================================
# LANGKAH 1: KUERI SQL JATAH RATA
# ==========================================
RENTANG_WAKTU = "1 year"

query_sql = f"""
    SELECT 
        waktu_aktual, 
        id_wilayah, 
        pm25, pm10, so2, co, no2, ozon
    FROM public.data_historis
    WHERE waktu_aktual >= NOW() - INTERVAL '{RENTANG_WAKTU}';
"""

ukuran_chunk = 50000
chunks = []

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = 0"))
    conn.commit() # Wajib commit agar pengaturannya tersimpan di sesi ini
    
    stream_conn = conn.execution_options(stream_results=True)
    
    for i, chunk in enumerate(pd.read_sql(text(query_sql), stream_conn, chunksize=ukuran_chunk)):
        print(f"✅ Berhasil menyedot batch ke-{i+1} (hingga {(i+1) * ukuran_chunk} baris...)")
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

kolom_waktu = 'waktu_aktual'
kolom_kota = 'id_wilayah'  
polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon'] 

# PRA-PEMROSESAN & RESAMPLE (PERUBAHAN KE JAM)
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])
df[kolom_waktu] = df[kolom_waktu].dt.tz_localize(None)
df.set_index(kolom_waktu, inplace=True)

# PENTING: Resample 'H' (Hourly/Jam), bukan 'D' (Daily/Harian)
df_hourly = df.groupby(kolom_kota)[polutan].resample('H').mean().reset_index()

# ==========================================
# LANGKAH 2: INTERPOLASI 
# ==========================================
print("Menambal data sensor yang bolong dengan interpolasi...")

df_hourly = df_hourly.groupby(kolom_kota, group_keys=False).apply(
    lambda group: group.interpolate(method='linear')
).reset_index(drop=True)

df_hourly = df_hourly.bfill().ffill()

print(f"Selesai! Bentuk data akhir (PER JAM): {df_hourly.shape}")
display(df_hourly.head())

Membuka jembatan ke Supabase...
✅ Berhasil menyedot batch ke-1 (hingga 50000 baris...)
✅ Berhasil menyedot batch ke-2 (hingga 100000 baris...)
✅ Berhasil menyedot batch ke-3 (hingga 150000 baris...)
✅ Berhasil menyedot batch ke-4 (hingga 200000 baris...)
✅ Berhasil menyedot batch ke-5 (hingga 250000 baris...)
✅ Berhasil menyedot batch ke-6 (hingga 300000 baris...)
✅ Berhasil menyedot batch ke-7 (hingga 350000 baris...)
Menambal data sensor yang bolong dengan interpolasi...
Selesai! Bentuk data akhir (PER JAM): (332994, 8)


,id_wilayah,waktu_aktual,pm25,pm10,so2,co,no2,ozon
0,1,2025-05-23 18:00:00,9.71,11.39,2.29,357.07,9.63,28.61
1,1,2025-05-23 19:00:00,11.67,14.05,2.93,447.67,13.06,25.51
2,1,2025-05-23 20:00:00,13.89,17.00,3.52,543.49,16.94,21.70
3,1,2025-05-23 21:00:00,15.57,19.10,3.77,615.73,19.99,18.70
4,1,2025-05-23 22:00:00,16.68,20.39,3.83,662.64,22.28,16.39


## Sel 3: Rekayasa Fitur

In [3]:
def create_advanced_features_24h(df_kota, n_lags=24, n_targets=24):
    df_temp = df_kota.copy()
    
    # FITUR TEMPORAL
    df_temp['Bulan'] = df_temp[kolom_waktu].dt.month
    df_temp['Jam'] = df_temp[kolom_waktu].dt.hour
    df_temp['Is_Weekend'] = df_temp[kolom_waktu].dt.dayofweek.isin([5, 6]).astype(int)

    for p in polutan:
        # FITUR HISTORY: Lihat 24 jam ke belakang
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
        # FITUR ROLLING: 72 Jam Terakhir (3 Hari)
        df_temp[f'{p}_RollMean_72'] = df_temp[p].rolling(window=72).mean()
        df_temp[f'{p}_RollMax_72'] = df_temp[p].rolling(window=72).max()
        
        # TARGET 24 JAM KE DEPAN (Multi-Output)
        for h in range(1, n_targets + 1):
            df_temp[f'TARGET_{p}_T+{h}'] = df_temp[p].shift(-h)
            
    return df_temp

print("Meracik fitur 24 Jam...")
df_model = df_hourly.groupby(kolom_kota, group_keys=False).apply(create_advanced_features_24h)
df_model = df_model.dropna().reset_index(drop=True)

df_model[kolom_kota] = df_model[kolom_kota].astype(str)
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

kolom_target = [col for col in df_model.columns if 'TARGET_' in col]
y = df_model[kolom_target]

kolom_bawaan = [kolom_waktu, 'kategori_ispu']
X = df_model.drop(columns=kolom_target + kolom_bawaan, errors='ignore')

print(f"Dimensi X (Soal AI)   : {X.shape}")
print(f"Dimensi y (Target 24J): {y.shape}")

# Sanity Check
assert X.isna().sum().sum() == 0, "Ada NaN di data X!"
assert y.isna().sum().sum() == 0, "Ada NaN di data y!"

display(X.head())
display(y.head())

Meracik fitur 24 Jam...
Dimensi X (Soal AI)   : (329384, 203)
Dimensi y (Target 24J): (329384, 144)


,pm25,pm10,so2,co,no2,ozon,Bulan,Jam,Is_Weekend,pm25_H-1,...,id_wilayah_35,id_wilayah_36,id_wilayah_37,id_wilayah_38,id_wilayah_4,id_wilayah_5,id_wilayah_6,id_wilayah_7,id_wilayah_8,id_wilayah_9
0,10.48,11.96,1.29,392.26,8.00,32.73,5,17,0,15.26,...,False,False,False,False,False,False,False,False,False,False
1,7.74,9.19,1.15,324.89,6.42,30.77,5,18,0,10.48,...,False,False,False,False,False,False,False,False,False,False
2,7.35,8.92,1.12,312.91,5.96,30.26,5,19,0,7.74,...,False,False,False,False,False,False,False,False,False,False
3,10.21,12.61,1.51,381.74,7.96,28.39,5,20,0,7.35,...,False,False,False,False,False,False,False,False,False,False
4,14.67,18.38,2.07,487.44,10.78,25.83,5,21,0,10.21,...,False,False,False,False,False,False,False,False,False,False


,TARGET_pm25_T+1,TARGET_pm25_T+2,TARGET_pm25_T+3,TARGET_pm25_T+4,TARGET_pm25_T+5,TARGET_pm25_T+6,TARGET_pm25_T+7,TARGET_pm25_T+8,TARGET_pm25_T+9,TARGET_pm25_T+10,...,TARGET_ozon_T+15,TARGET_ozon_T+16,TARGET_ozon_T+17,TARGET_ozon_T+18,TARGET_ozon_T+19,TARGET_ozon_T+20,TARGET_ozon_T+21,TARGET_ozon_T+22,TARGET_ozon_T+23,TARGET_ozon_T+24
0,7.74,7.35,10.21,14.67,17.97,20.13,21.76,22.62,22.13,22.48,...,30.55,38.14,52.88,71.96,93.07,110.50,113.10,100.25,89.44,75.49
1,7.35,10.21,14.67,17.97,20.13,21.76,22.62,22.13,22.48,22.57,...,38.14,52.88,71.96,93.07,110.50,113.10,100.25,89.44,75.49,67.63
2,10.21,14.67,17.97,20.13,21.76,22.62,22.13,22.48,22.57,22.25,...,52.88,71.96,93.07,110.50,113.10,100.25,89.44,75.49,67.63,62.92
3,14.67,17.97,20.13,21.76,22.62,22.13,22.48,22.57,22.25,21.99,...,71.96,93.07,110.50,113.10,100.25,89.44,75.49,67.63,62.92,60.24
4,17.97,20.13,21.76,22.62,22.13,22.48,22.57,22.25,21.99,22.22,...,93.07,110.50,113.10,100.25,89.44,75.49,67.63,62.92,60.24,58.92


## Sel 4: Splitting & Training Baseline Model MLFLOW

### Splitting

In [ ]:
# 1. Splitting Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Mengecilkan ukuran data di RAM (Downcasting ke Float32)...")
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

# ==========================================
# STRATEGI BARU: PROXY TASK (1 TARGET SAJA)
# ==========================================
print("Memotong target menjadi 1 kolom saja (Proxy Task) untuk adu mekanik Baseline...")
# Mengambil kolom target pertama saja (indeks 0). Support untuk format Pandas maupun Numpy.
if hasattr(y_train, 'iloc'):
    y_train_proxy = y_train.iloc[:, 0]
    y_test_proxy = y_test.iloc[:, 0]
else:
    y_train_proxy = y_train[:, 0]
    y_test_proxy = y_test[:, 0]

### Tracking MLFlow DagsHub

In [ ]:
# ==========================================
# TRACKING MLFLOW DENGAN DAGSHUB
# ==========================================
# 2. Ambil Konfigurasi DagsHub dari .env
repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
repo_name = os.getenv("DAGSHUB_REPO_NAME")

if not repo_owner or not repo_name:
    raise ValueError("Identitas DAGSHUB_REPO_OWNER atau DAGSHUB_REPO_NAME belum di-set di file .env!")

print(f"Connecting to DagsHub MLflow Server ({repo_owner}/{repo_name})...")
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)

In [ ]:
# 3. Set Eksperimen Tahap 1
import gc # Library untuk Garbage Collection (Tukang Sapu RAM)

mlflow.set_experiment("Baseline_Selection_Tegar")

# 4. Persiapan Baseline Model (MURNI SINGLE OUTPUT)
baselines = {
    # Tanpa MultiOutputRegressor, langsung model murni!
    "Baseline_DT_Single": lambda: DecisionTreeRegressor(max_depth=5, random_state=42),
    "Baseline_RF_Single": lambda: RandomForestRegressor(n_estimators=20, max_depth=5, random_state=42),
    "Baseline_XGB_Single": lambda: xgb.XGBRegressor(n_estimators=20, max_depth=5, random_state=42, tree_method="hist", device="cuda")
}

print("--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---")
for nama_model, cetak_biru_model in baselines.items():
    with mlflow.start_run(run_name=f"{nama_model}_Run_Tegar"):

        model_aktif = cetak_biru_model()

        print(f"Sedang melatih {nama_model} (Pada 1 Target Proxy)...")
        # Masukkan versi proxy (1 kolom) ke dalam mesin
        model_aktif.fit(X_train, y_train_proxy)
        y_pred_proxy = model_aktif.predict(X_test)

        # Kalkulasi Metrik (Hanya untuk 1 target tersebut)
        mae = mean_absolute_error(y_test_proxy, y_pred_proxy)
        rmse = np.sqrt(mean_squared_error(y_test_proxy, y_pred_proxy))
        r2 = r2_score(y_test_proxy, y_pred_proxy)

        # Mencatat parameter, metrik, dan artifact model ke MLflow
        mlflow.log_param("algoritma", nama_model)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)

        print(f"--> {nama_model} berhasil dicatat (RMSE: {rmse:.2f})")
        
        # Hapus model dan prediksi dari memori 
        del model_aktif
        del y_pred_proxy
        gc.collect() 
        print(f"Memori dibersihkan. Lanjut ke model berikutnya...\n")

Connecting to DagsHub MLflow Server (tegarkusuma12/Web-ISPU)...


Initialized MLflow to track repo "tegarkusuma12/Web-ISPU"

Repository tegarkusuma12/Web-ISPU initialized!

--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---
